# Phase 1: Exact Post-Conv / Post-$\mu$P `FalconH1Mixer` State Hook & Canonicalization

This notebook contains the complete Phase 1 implementation and verified outputs for the CoD (`hybrid_state_eviction_v13`) pipeline:
- **Cell 1.0:** Setup, directory paths, environment report, and module check
- **Cell 1.1:** Patch Phase 0 loader for optional single-GPU `device_map` override
- **Cell 1.2:** Implement `hooks/falcon_h1_state_hook.py` with exact mixer front-end replay and unit-key canonicalization
- **Cell 1.3:** Implement `hooks/phase1_checks.py` containing dual verification (Test 1: Readout identity, Test 2: State reconstruction & stepwise recurrence) and stability assertions
- **Cell 1.4:** Update `hooks/__init__.py`, purge stale imports, and verify module signatures
- **Cell 1.5:** Fast smoke test on `Falcon-H1-0.5B` ($P=512$) verifying zero passive interference on logits
- **Cell 1.6:** Full run with $P=8,192$ and $P=2,048$ captures on both `0.5B` and `1.5B-Deep` models
- **Cell 1.7:** Phase 1 gate audit, summary JSON verification, and handover manifest generation (`checkpoints/phase1/phase1_manifest.json`)

### Cell 1.0 - Setup, Paths & Environment Report | Phase 1: State Hook & Canonicalization

In [1]:
#Cell 1.0 - Setup, Paths & Environment Report | Phase 1: State Hook & Canonicalization
import os, sys, json, gc, time, platform
from pathlib import Path
import torch
import transformers

PROJECT_ROOT = Path("/kaggle/working/hybrid_state_eviction_v13")
CFG_DIR = PROJECT_ROOT / "configs"
PHASE1_DIR = PROJECT_ROOT / "checkpoints" / "phase1"      # FIXED handover path for Phase 2
PHASE1_DIR.mkdir(parents=True, exist_ok=True)
FORCE_RERUN = False        # True -> ignore saved report_*.json / ref captures and recompute

for y in ("falcon_h1_0.5b.yaml", "falcon_h1_1.5b_deep.yaml"):
    assert (CFG_DIR / y).exists(), f"Missing {y}: run Phase 0 cells 0.0-0.2 first."
assert (PROJECT_ROOT / "hooks" / "phase0_core.py").exists(), "Run Phase 0 Cell 0.2 first."
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"python {platform.python_version()} | torch {torch.__version__} | transformers {transformers.__version__}")
print(f"cuda available: {torch.cuda.is_available()} | gpus: {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}")
for mod in ("mamba_ssm", "causal_conv1d"):
    try:
        __import__(mod); print(f"WARNING: {mod} IS installed -> numerics path differs from the reference path the hook replays")
    except Exception:
        print(f"{mod}: not installed (expected; reference torch path)")
print(f"[Phase 1.0] handover dir: {PHASE1_DIR}")

python 3.10.12 | torch 2.4.0 | transformers 4.44.2
cuda available: True | gpus: ['Tesla T4', 'Tesla T4']
mamba_ssm: not installed (expected; reference torch path)
causal_conv1d: not installed (expected; reference torch path)
[Phase 1.0] handover dir: /kaggle/working/hybrid_state_eviction_v13/checkpoints/phase1

### Cell 1.1 - Patch Phase 0 Loader: Optional Single-GPU `device_map` | Phase 1 (edit of Phase 0 `hooks/phase0_core.py`)

In [2]:
# ==============================================================================
# Cell 1.1 - Patch Phase 0 Loader: Optional Single-GPU `device_map` | Phase 1 (edit of Phase 0 `hooks/phase0_core.py`)
# Idempotent. Adds `device_map=None` argument to `load_falcon_h1_for_t4` (default behaviour unchanged).
# ==============================================================================
from pathlib import Path
PROJECT_ROOT = Path("/kaggle/working/hybrid_state_eviction_v13")
p = PROJECT_ROOT / "hooks" / "phase0_core.py"
txt = p.read_text()

if "device_map: Optional[Any] = None" in txt:
    print("[Phase 1.1] phase0_core.py already patched.")
else:
    old_sig = ("def load_falcon_h1_for_t4(\n    yaml_path: Union[str, Path],\n    load_tokenizer: bool = True,\n)")
    new_sig = ("def load_falcon_h1_for_t4(\n    yaml_path: Union[str, Path],\n    load_tokenizer: bool = True,\n"
               "    device_map: Optional[Any] = None,\n)")
    old_dm = '        device_map=cfg["precision_guard"].get("device_map", "auto"),'
    new_dm = ('        device_map=(device_map if device_map is not None '
              'else cfg["precision_guard"].get("device_map", "auto")),')
    assert txt.count(old_sig) == 1 and txt.count(old_dm) == 1, "phase0_core.py differs from expected; paste it to me."
    p.write_text(txt.replace(old_sig, new_sig).replace(old_dm, new_dm))
    print("[Phase 1.1] Patched phase0_core.py (device_map override).")

[Phase 1.1] Patched phase0_core.py (device_map override).

### Cell 1.2 - Write hooks/falcon_h1_state_hook.py | Phase 1: State Hook & Canonicalization

In [3]:
#Cell 1.2 - Write hooks/falcon_h1_state_hook.py | Phase 1: State Hook & Canonicalization
from pathlib import Path
PROJECT_ROOT = Path("/kaggle/working/hybrid_state_eviction_v13")
HOOK_CODE = r'''"""Phase 1: Exact post-conv / post-muP FalconH1Mixer state hook & unit-key canonicalization.

Handover contract to Phase 2 (`hooks/survival_metrics.py`), per mixer layer l (dict key = layer_idx):
    b_hat     [B, P, d_state]            unit-norm write key  (n_groups == 1)
    u         [B, P, H_ssm, d_head_ssm]  effective value vector
    w         [B, P, H_ssm]              write energy ||u||_2
    log_alpha [B, P, H_ssm]              l_{t,m}(P) = sum_{tau=t+1}^{P-1} log a_{tau,m}  (<= 0)
    alpha     [B, P, H_ssm]              exp(log_alpha)  in [0, 1]
    S_P       [B, H_ssm, d_state, d_head_ssm]  terminal prefill state from the model cache (canonical layout)
Extras (additive, do not break the contract): delta [B,P,H], b_norm [B,P], S_P_cache_layout (str).
"""

from __future__ import annotations

from typing import Any, Dict, List, Optional, Sequence

import torch
import torch.nn.functional as F

from .phase0_core import _resolve_ssm_tensor

EPS = 1e-8


def canonical_state(S: torch.Tensor, d_head: int, d_state: int) -> torch.Tensor:
    """Return state as [B, H, d_state, d_head] fp32. Cache layout seen on Kaggle is [B, H, d_head, d_state]."""
    if S.shape[-2:] == (d_head, d_state):
        return S.transpose(-1, -2).contiguous().float()
    if S.shape[-2:] == (d_state, d_head):
        return S.contiguous().float()
    raise ValueError(f"Unexpected SSM state shape {tuple(S.shape)} for d_head={d_head}, d_state={d_state}")


def _cache_layer_has_state(cache: Any, layer_idx: int) -> bool:
    try:
        return _resolve_ssm_tensor(cache.layers[layer_idx]) is not None
    except Exception:
        return False


class FalconH1StateHook:
    """Forward-pre-hook on every `FalconH1Mixer`; replays the mixer front-end op-for-op to expose the exact
    tensors the selective scan consumes, then canonicalizes them. Single-shot prefill only (fresh cache)."""

    def __init__(self, model, store_device: Any = "same", eps: float = EPS, layer_ids: Optional[Sequence[int]] = None):
        self.model = model
        self.eps = float(eps)
        self.store_device = store_device  # "same" | "cpu" | "cuda:0" ...
        self.mixers = self._find_mixers(model)
        if layer_ids is not None:
            keep = set(int(i) for i in layer_ids)
            self.mixers = [(i, m) for i, m in self.mixers if i in keep]
        self.records: Dict[int, Dict[str, Any]] = {}
        self._handles: List[Any] = []

    @staticmethod
    def _find_mixers(model):
        backbone = getattr(model, "model", model)
        out = []
        for i, layer in enumerate(backbone.layers):
            mixer = getattr(layer, "mamba", None)
            if mixer is None:
                mixer = getattr(layer, "mixer", None)
            if mixer is None:
                continue
            out.append((int(getattr(mixer, "layer_idx", i)), mixer))
        if not out:
            raise RuntimeError("No FalconH1Mixer found (expected `model.model.layers[i].mamba`).")
        return out

    # ------------------------------------------------------------------ hook plumbing
    def attach(self):
        if self._handles:
            return self
        for layer_idx, mixer in self.mixers:
            self._handles.append(mixer.register_forward_pre_hook(self._make_pre_hook(layer_idx), with_kwargs=True))
        return self

    def detach(self):
        for h in self._handles:
            h.remove()
        self._handles = []

    def __enter__(self):
        return self.attach()

    def __exit__(self, *exc):
        self.detach()

    def release(self):
        self.records = {}

    def _make_pre_hook(self, layer_idx: int):
        def hook(mixer, args, kwargs):
            hs = kwargs.get("hidden_states", args[0] if len(args) > 0 else None)
            cache = kwargs.get("cache_params", None)
            mask = kwargs.get("attention_mask", None)
            if hs is None:
                return None
            if cache is not None and _cache_layer_has_state(cache, layer_idx):
                raise RuntimeError(
                    "FalconH1StateHook supports single-shot prefill on a fresh cache only "
                    "(conv/recurrent state already present -> chunked prefill)."
                )
            self.records[layer_idx] = self._extract(mixer, hs, mask)
            return None  # do not alter the forward call

        return hook

    def _to_store(self, t: torch.Tensor) -> torch.Tensor:
        if self.store_device == "same":
            return t
        return t.to(self.store_device)

    # ------------------------------------------------------------------ exact extraction
    @torch.no_grad()
    def _extract(self, mixer, hidden_states: torch.Tensor, attention_mask: Optional[torch.Tensor]) -> Dict[str, Any]:
        # ==============================================================================
        # [MATH-UPGRADE | Phase 1 -> Phase 2]
        # Original Spec: conv1d -> SiLU on xBC_raw, THEN apply muP ssm_multipliers (mu_x, mu_B, mu_C, mu_dt).
        # Upgraded Spec: Real FalconH1Mixer applies `mup_vector` to the in_proj OUTPUT *before* conv1d:
        #                proj = in_proj(ssm_in_multiplier * h) * mup_vector ; xBC = SiLU(conv1d(proj_xBC)).
        #                muP is therefore already inside the post-conv (x, B, C); dt = softplus(proj_dt + dt_bias)
        #                (mu_dt already inside proj_dt), clamped to time_step_limit. No post-conv multiplier exists.
        #                The front-end is replayed op-for-op (same dtypes) so the tensors are bit-identical to what
        #                `mamba2_chunk_scan` consumes; all later math is fp32 (reverse cumsum in fp64).
        # Reason:        Conv+SiLU is nonlinear, so (mu * SiLU(conv(z))) != SiLU(conv(mu * z)); the plan's order
        #                would give wrong x, B, Delta and break the S_P identity.
        # Contract Check: Output keys/shapes/dtypes identical to Phase 1 spec (fp32); extras only added.
        # ==============================================================================
        eps = self.eps
        bsz, L, _ = hidden_states.shape
        H, hd, N, G = mixer.num_heads, mixer.head_dim, mixer.ssm_state_size, mixer.n_groups
        if G != 1:
            raise NotImplementedError("Phase 1 hook assumes n_groups == 1 (true for Falcon-H1 0.5B / 1.5B-Deep).")
        I, conv_dim = mixer.intermediate_size, mixer.conv_dim

        h = hidden_states
        if attention_mask is not None:
            h = (h * attention_mask[:, :, None]).to(h.dtype)
        h = h * mixer.ssm_in_multiplier
        proj = mixer.in_proj(h)
        proj = proj * mixer.mup_vector
        _gate, xBC, dt_raw = proj.split([I, conv_dim, H], dim=-1)

        xBC_t = xBC.transpose(1, 2)
        w = mixer.conv1d.weight.squeeze(1)
        conv = F.conv1d(
            xBC_t.to(w.dtype), w.unsqueeze(1), mixer.conv1d.bias,
            padding=w.shape[-1] - 1, groups=conv_dim,
        )[:, :, :L]
        conv = getattr(mixer, "act", F.silu)(conv).to(xBC_t.dtype)
        xBC_c = conv.transpose(1, 2)
        if attention_mask is not None:
            xBC_c = (xBC_c * attention_mask[:, :, None]).to(xBC_c.dtype)
        x, B, _C = torch.split(xBC_c, [I, G * N, G * N], dim=-1)

        dt = dt_raw + mixer.dt_bias.to(dt_raw.dtype)
        dt = F.softplus(dt)
        lim = getattr(mixer, "time_step_limit", None)
        if lim is not None:
            dt = torch.clamp(dt, min=lim[0], max=lim[1])

        # ---- fp32 canonicalization ------------------------------------------------
        delta = dt.float()                                            # [B,P,H]  > 0
        A_pos = torch.exp(mixer.A_log.float())                        # [H]      > 0
        log_a = -(delta * A_pos.view(1, 1, H))                        # [B,P,H]  <= 0
        x32 = x.float().reshape(bsz, L, H, hd)
        B32 = B.float().reshape(bsz, L, N)
        b_norm = torch.linalg.vector_norm(B32, dim=-1)                # [B,P]
        denom = b_norm + eps
        b_hat = B32 / denom.unsqueeze(-1)                             # [B,P,N]
        # u = Delta * (||B|| + eps) * x  =>  b_hat^T u == B^T (Delta x) identically (plan 1.4)
        u = (delta * denom.unsqueeze(-1)).unsqueeze(-1) * x32         # [B,P,H,hd]
        wgt = torch.linalg.vector_norm(u, dim=-1)                     # [B,P,H]

        # ==============================================================================
        # [MATH-UPGRADE | Phase 1 -> Phase 2]
        # Original Spec: l_t = c_{P-1} - c_t from an fp32 inclusive cumsum c.
        # Upgraded Spec: exclusive REVERSE cumsum in fp64: l_t = rev_inclusive_cumsum_t - log a_t.
        # Reason:        fp32 subtraction of two large cumulative sums loses ~1e-7*|c| absolute accuracy in l_t
        #                (alpha relative error ~1e-4 for |c|~1e3); fp64 removes it at negligible cost (P*H elements).
        # Contract Check: log_alpha/alpha returned as fp32 [B,P,H]; l_{P-1}=0, alpha_{P-1}=1 exactly.
        # ==============================================================================
        la64 = log_a.double()
        rev = torch.flip(torch.cumsum(torch.flip(la64, dims=[1]), dim=1), dims=[1])
        ell64 = rev - la64
        log_alpha = ell64.float()
        alpha = torch.exp(ell64).float()

        rec = {
            "b_hat": b_hat, "u": u, "w": wgt, "log_alpha": log_alpha, "alpha": alpha,
            "delta": delta, "b_norm": b_norm,
        }
        return {k: self._to_store(v.contiguous()) for k, v in rec.items()}

    # ------------------------------------------------------------------ one-call capture
    @torch.no_grad()
    def capture(self, input_ids: torch.Tensor):
        """Run ONE fresh prefill; returns (records, cache, last_logits). Adds `S_P` (from the real cache)."""
        self.records = {}
        with self:
            out = self.model(input_ids=input_ids, use_cache=True, logits_to_keep=1)
        cache = out.past_key_values
        for layer_idx, mixer in self.mixers:
            resolved = _resolve_ssm_tensor(cache.layers[layer_idx])
            if resolved is None:
                raise RuntimeError(f"No SSM state found in cache for layer {layer_idx}.")
            S = canonical_state(resolved[2], mixer.head_dim, mixer.ssm_state_size)
            self.records[layer_idx]["S_P"] = self._to_store(S)
        return self.records, cache, out.logits

    @staticmethod
    def records_nbytes(records: Dict[int, Dict[str, Any]]) -> int:
        return sum(t.numel() * t.element_size() for r in records.values() for t in r.values() if torch.is_tensor(t))
'''
(PROJECT_ROOT / "hooks" / "falcon_h1_state_hook.py").write_text(HOOK_CODE)
print("[Phase 1.2] wrote hooks/falcon_h1_state_hook.py")

[Phase 1.2] wrote hooks/falcon_h1_state_hook.py

### Cell 1.3 - Write hooks/phase1_checks.py (Tests 1, 2, Stability) | Phase 1: State Hook & Canonicalization

In [4]:
#Cell 1.3 - Write hooks/phase1_checks.py (Tests 1, 2, Stability) | Phase 1: State Hook & Canonicalization
from pathlib import Path
PROJECT_ROOT = Path("/kaggle/working/hybrid_state_eviction_v13")
CHECKS_CODE = r'''"""Phase 1 verification utilities: Test 1 (readout identity), Test 2 (recurrence vs cache), stability (Test 9 lite)."""

from __future__ import annotations

import time
from typing import Any, Dict, List, Optional, Sequence

import torch

TEST2_TOL = 1e-4      # relative Frobenius error vs cache state (plan Test 2)
TEST1_TOL = 1e-5      # abs error, scaled by max(1, max|readout|) (plan Test 1)


def _rel_fro(a: torch.Tensor, b: torch.Tensor) -> float:
    """Relative Frobenius error ||a-b||_F / ||b||_F over all elements (fp64)."""
    a, b = a.double(), b.double()
    return float((a - b).flatten().norm() / (b.flatten().norm() + 1e-30))


def reconstruct_terminal_state(rec: Dict[str, torch.Tensor], device=None, dtype=torch.float64) -> torch.Tensor:
    """S_P = sum_t alpha_t b_hat_t^T u_t  -> [B,H,N,hd] (closed-form decomposition)."""
    dev = device or rec["u"].device
    a = rec["alpha"].to(dev, dtype)                     # [B,P,H]
    b = rec["b_hat"].to(dev, dtype)                     # [B,P,N]
    u = rec["u"].to(dev, dtype)                         # [B,P,H,hd]
    au = a.unsqueeze(-1) * u                            # [B,P,H,hd]
    return torch.einsum("bpn,bphd->bhnd", b, au)


@torch.no_grad()
def stepwise_recurrence(rec: Dict[str, torch.Tensor], A_log: torch.Tensor, device=None) -> torch.Tensor:
    """Independent fp32 step-by-step recurrence S_t = a_t S_{t-1} + b_hat_t^T u_t, a_t = exp(-Delta_t A_m).
    Uses delta and A_log directly (NOT the cumsum path), so it genuinely cross-checks the decomposition."""
    dev = device or rec["u"].device
    b = rec["b_hat"][0].to(dev, torch.float32)          # [P,N]
    u = rec["u"][0].to(dev, torch.float32)              # [P,H,hd]
    delta = rec["delta"][0].to(dev, torch.float32)      # [P,H]
    a = torch.exp(-(delta * torch.exp(A_log.to(dev, torch.float32)).unsqueeze(0)))   # [P,H]
    P, H, hd = u.shape
    N = b.shape[-1]
    S = torch.zeros(H, N, hd, device=dev, dtype=torch.float32)
    for t in range(P):
        S.mul_(a[t].view(H, 1, 1)).add_(b[t].view(1, N, 1) * u[t].view(H, 1, hd))
    return S.unsqueeze(0)


@torch.no_grad()
def readout_identity_error(rec: Dict[str, torch.Tensor], n_probe: int = 128, seed: int = 0, device=None) -> Dict[str, float]:
    """Test 1: b_hat_t S_P  ==  sum_tau alpha_tau <b_hat_t, b_hat_tau> u_tau   (all fp32), probed on n_probe positions."""
    dev = device or rec["u"].device
    b = rec["b_hat"][0].to(dev, torch.float32)          # [P,N]
    u = rec["u"][0].to(dev, torch.float32)              # [P,H,hd]
    a = rec["alpha"][0].to(dev, torch.float32)          # [P,H]
    P = b.shape[0]
    g = torch.Generator().manual_seed(seed)
    idx = torch.randperm(P, generator=g)[: min(n_probe, P)].to(dev)
    S = reconstruct_terminal_state(rec, device=dev, dtype=torch.float32)[0]          # [H,N,hd]
    lhs = torch.einsum("tn,hnd->thd", b[idx], S)                                      # [T,H,hd]
    gram = b[idx] @ b.T                                                               # [T,P]
    rhs = torch.einsum("tp,phd->thd", gram, a.unsqueeze(-1) * u)                      # [T,H,hd]
    err = float((lhs - rhs).abs().max())
    scale = float(max(1.0, lhs.abs().max()))
    # cross-check against the *real* cache state as well (informational)
    S_cache = rec["S_P"][0].to(dev, torch.float32)
    lhs_cache = torch.einsum("tn,hnd->thd", b[idx], S_cache)
    err_cache_rel = float((lhs_cache - rhs).abs().max() / (lhs_cache.abs().max() + 1e-30))
    return {"max_abs_err": err, "scale": scale, "scaled_err": err / scale, "cache_readout_rel_err": err_cache_rel}


@torch.no_grad()
def verify_capture(
    records: Dict[int, Dict[str, torch.Tensor]],
    mixers: Sequence,                      # [(layer_idx, mixer)]
    step_layers: Optional[Sequence[int]] = None,   # None -> all layers
    test1_layers: Sequence[int] = (),
    logits: Optional[torch.Tensor] = None,
    compute_device: Optional[Any] = None,
) -> Dict[str, Any]:
    """Run Test 2 (decomposition AND stepwise vs cache), Test 1 (subset), and finiteness / range checks."""
    A_logs = {i: m.A_log.detach() for i, m in mixers}
    layer_rows: List[Dict[str, Any]] = []
    t0 = time.perf_counter()
    all_finite = True
    ranges_ok = True
    for i in sorted(records):
        rec = records[i]
        dev = compute_device or rec["u"].device
        S_cache = rec["S_P"].to(dev, torch.float64)
        S_dec = reconstruct_terminal_state(rec, device=dev, dtype=torch.float64)
        row: Dict[str, Any] = {"layer": i, "rel_fro_decomp_vs_cache": _rel_fro(S_dec, S_cache)}
        if step_layers is None or i in step_layers:
            S_step = stepwise_recurrence(rec, A_logs[i], device=dev)
            row["rel_fro_step_vs_cache"] = _rel_fro(S_step.double(), S_cache)
            row["rel_fro_step_vs_decomp"] = _rel_fro(S_step.double(), S_dec)
        if i in test1_layers:
            row["test1"] = readout_identity_error(rec, device=dev)
        fin = all(bool(torch.isfinite(v).all()) for v in rec.values() if torch.is_tensor(v))
        all_finite &= fin
        la, al, wg, bh = rec["log_alpha"], rec["alpha"], rec["w"], rec["b_hat"]
        rng = bool((la <= 1e-6).all() and (al >= 0).all() and (al <= 1 + 1e-6).all() and (wg >= 0).all())
        nrm = bh.norm(dim=-1)
        rng &= bool(((nrm - 1).abs() < 1e-4).all())
        rng &= bool(la[:, -1].abs().max() == 0)           # l_{P-1} == 0
        ranges_ok &= rng
        row.update({"finite": fin, "ranges_ok": rng,
                    "alpha_min": float(al.min()), "log_alpha_min": float(la.min()),
                    "S_P_fro": float(S_cache.norm()), "w_mean": float(wg.mean())})
        layer_rows.append(row)
    logits_finite = True if logits is None else bool(torch.isfinite(logits).all())

    dec = [r["rel_fro_decomp_vs_cache"] for r in layer_rows]
    stp = [r["rel_fro_step_vs_cache"] for r in layer_rows if "rel_fro_step_vs_cache" in r]
    t1 = [r["test1"]["scaled_err"] for r in layer_rows if "test1" in r]
    summary = {
        "n_layers": len(layer_rows),
        "test2_decomp_max": max(dec), "test2_decomp_median": float(torch.tensor(dec).median()),
        "test2_step_max": max(stp) if stp else None,
        "test1_scaled_err_max": max(t1) if t1 else None,
        "all_finite": bool(all_finite and logits_finite),
        "ranges_ok": bool(ranges_ok),
        "verify_seconds": round(time.perf_counter() - t0, 2),
    }
    summary["test2_pass"] = bool(summary["test2_decomp_max"] < TEST2_TOL and (not stp or summary["test2_step_max"] < TEST2_TOL))
    summary["test1_pass"] = bool((not t1) or summary["test1_scaled_err_max"] < TEST1_TOL)
    summary["stability_pass"] = bool(summary["all_finite"] and summary["ranges_ok"])
    summary["phase1_pass"] = bool(summary["test2_pass"] and summary["test1_pass"] and summary["stability_pass"])
    return {"summary": summary, "layers": layer_rows}


def make_input_ids(kind: str, P: int, vocab_size: int, tokenizer=None, seed: int = 0, device="cuda:0") -> torch.Tensor:
    """kind='random' -> uniform ids; kind='text' -> structured pseudo-document (needs tokenizer)."""
    if kind == "text":
        assert tokenizer is not None, "tokenizer required for kind='text'"
        parts, i = [], 0
        while True:
            parts.append(f"Archive entry {i}: the access code for vault {(i * 7) % 97} is {(i * 7919) % 100000:05d}. "
                         f"The keeper of room {(i * 13) % 53} noted that sector {(i * 31) % 211} was inspected. ")
            i += 1
            if i % 64 == 0:
                ids = tokenizer("".join(parts), return_tensors="pt").input_ids
                if ids.shape[1] >= P:
                    return ids[:, :P].to(device)
    g = torch.Generator().manual_seed(seed)
    return torch.randint(10, min(vocab_size, 30000), (1, P), generator=g).to(device)
'''
(PROJECT_ROOT / "hooks" / "phase1_checks.py").write_text(CHECKS_CODE)
print("[Phase 1.3] wrote hooks/phase1_checks.py")

[Phase 1.3] wrote hooks/phase1_checks.py

### Cell 1.4 - Update hooks/__init__.py, Purge Stale Imports, Import Check | Phase 1: State Hook & Canonicalization

In [5]:
#Cell 1.4 - Update hooks/__init__.py, Purge Stale Imports, Import Check | Phase 1: State Hook & Canonicalization
import sys, importlib
from pathlib import Path
PROJECT_ROOT = Path("/kaggle/working/hybrid_state_eviction_v13")

(PROJECT_ROOT / "hooks" / "__init__.py").write_text(
    '"""Hooks package: Phase 0 T4 guards + Phase 1 state hook."""\n'
    "from .phase0_core import (\n"
    "    load_model_config, sync_arch_config_with_hf_config, verify_memory_accounting,\n"
    "    compute_retained_kv_ratio, apply_t4_fp32_ssm_guard, load_falcon_h1_for_t4,\n"
    "    ensure_unified_cache_views, force_ssm_states_fp32,\n"
    ")\n"
    "from .falcon_h1_state_hook import FalconH1StateHook, canonical_state\n"
    "from .phase1_checks import (\n"
    "    verify_capture, make_input_ids, reconstruct_terminal_state,\n"
    "    stepwise_recurrence, readout_identity_error,\n"
    ")\n"
)
for m in [m for m in sys.modules if m == "hooks" or m.startswith("hooks.")]:
    del sys.modules[m]
importlib.invalidate_caches()

from hooks import FalconH1StateHook, verify_capture, make_input_ids, load_falcon_h1_for_t4
import inspect
assert "device_map" in inspect.signature(load_falcon_h1_for_t4).parameters, "Re-run Cell 1.1"
print("[Phase 1.4] imports OK")

[Phase 1.4] imports OK

### Cell 1.5 - SMOKE TEST (0.5B, P=512, all layers) | Phase 1: State Hook & Canonicalization

In [6]:
#Cell 1.5 - SMOKE TEST (0.5B, P=512, all layers) | Phase 1: State Hook & Canonicalization
import gc, json, time, torch
from pathlib import Path
from hooks.phase0_core import load_falcon_h1_for_t4
from hooks.falcon_h1_state_hook import FalconH1StateHook
from hooks.phase1_checks import verify_capture, make_input_ids

PROJECT_ROOT = Path("/kaggle/working/hybrid_state_eviction_v13")
PHASE1_DIR = PROJECT_ROOT / "checkpoints" / "phase1"
PHASE1_DIR.mkdir(parents=True, exist_ok=True)

model, tok, cfg = load_falcon_h1_for_t4(PROJECT_ROOT / "configs" / "falcon_h1_0.5b.yaml",
                                        load_tokenizer=True, device_map={"": 0})
L = cfg["architecture"]["num_hidden_layers"]
P = 512
ids = make_input_ids("text", P, model.config.vocab_size, tok, device="cuda:0")

with torch.no_grad():                                   # reference logits WITHOUT the hook
    ref_logits = model(input_ids=ids, use_cache=True, logits_to_keep=1).logits

hook = FalconH1StateHook(model, store_device="same")
t0 = time.perf_counter()
records, cache, logits = hook.capture(ids)
torch.cuda.synchronize()
cap_s = time.perf_counter() - t0
passive = float((ref_logits.float() - logits.float()).abs().max())

rep = verify_capture(records, hook.mixers, step_layers=None,
                     test1_layers=sorted({0, L // 2, L - 1}), logits=logits, compute_device="cuda:0")
s = rep["summary"]

m0 = hook.mixers[0][1]
r0 = records[0]
print("--- environment facts the hook relies on ---")
print(f"mup_vector dtype      : {m0.mup_vector.dtype} | ssm_multipliers: {model.config.ssm_multipliers} | ssm_in_multiplier: {model.config.ssm_in_multiplier}")
print(f"time_step_limit       : {getattr(m0, 'time_step_limit', None)} | n_groups: {m0.n_groups} | conv_dim: {m0.conv_dim}")
print(f"record shapes (L0)    : b_hat {tuple(r0['b_hat'].shape)} u {tuple(r0['u'].shape)} w {tuple(r0['w'].shape)} "
      f"alpha {tuple(r0['alpha'].shape)} S_P {tuple(r0['S_P'].shape)}")
print(f"records size          : {hook.records_nbytes(records)/1e6:.1f} MB | capture {cap_s:.2f}s")
print(f"hook passive (max |logit diff| with vs without hook): {passive:.3e}")
print("--- summary ---")
print(json.dumps(s, indent=2))
worst = sorted(rep["layers"], key=lambda r: -r["rel_fro_decomp_vs_cache"])[:5]
print("worst 5 layers (decomp vs cache):", [(r["layer"], f"{r['rel_fro_decomp_vs_cache']:.2e}",
      f"{r.get('rel_fro_step_vs_cache', float('nan')):.2e}") for r in worst])
print("test1 samples:", [(r["layer"], {k: round(v, 8) for k, v in r["test1"].items()}) for r in rep["layers"] if "test1" in r])

(PHASE1_DIR / "smoke_report.json").write_text(json.dumps({"summary": s, "passive_logit_diff": passive,
                                                           "layers": rep["layers"]}, indent=1))
ok = s["phase1_pass"] and passive <= 1e-3
print("\n" + ("SMOKE PASS" if ok else "SMOKE FAIL"))
hook.detach(); del hook, records, cache, logits, ref_logits, model
gc.collect(); torch.cuda.empty_cache()
if not ok:
    raise SystemExit("Smoke failed -> send me this cell's full output (do NOT run 1.6).")

tokenizer_config.json: 100%
 99.9k/99.9k [00:00<00:00, 10.5MB/s]
tokenizer.json: 100%
 2.35M/2.35M [00:00<00:00, 47.2MB/s]
special_tokens_map.json: 100%
 7.69k/7.69k [00:00<00:00, 860kB/s]
chat_template.jinja: 100%
 1.15k/1.15k [00:00<00:00, 133kB/s]
Loading weights: 100%
 579/579 [00:00<00:00, 1188.50it/s]
--- environment facts the hook relies on ---
mup_vector dtype      : torch.float16 | ssm_multipliers: [0.3535533905932738, 0.25, 0.3535533905932738, 0.5, 0.3535533905932738] | ssm_in_multiplier: 1.25
time_step_limit       : (0.0, inf) | n_groups: 1 | conv_dim: 1792
record shapes (L0)    : b_hat (1, 512, 128) u (1, 512, 24, 64) w (1, 512, 24) alpha (1, 512, 24) S_P (1, 24, 128, 64)
records size          : 158.1 MB | capture 0.24s
hook passive (max |logit diff| with vs without hook): 0.000e+00
--- summary ---
{
  "n_layers": 36,
  "test2_decomp_max": 2.874617742270267e-06,
  "test2_decomp_median": 6.24346967015299e-07,
  "test2_step_max": 6.066701415961222e-06,
  "test1_scaled_err_max

### Cell 1.6 - FULL RUN: P=8192 Capture + Verification (0.5B then 1.5B-Deep) + Reference Captures | Phase 1: State Hook & Canonicalization

In [7]:
#Cell 1.6 - FULL RUN: P=8192 Capture + Verification (0.5B then 1.5B-Deep) + Reference Captures | Phase 1: State Hook & Canonicalization
import gc, json, time, torch
from pathlib import Path
from hooks.phase0_core import load_falcon_h1_for_t4
from hooks.falcon_h1_state_hook import FalconH1StateHook
from hooks.phase1_checks import verify_capture, make_input_ids

PROJECT_ROOT = Path("/kaggle/working/hybrid_state_eviction_v13")
CFG_DIR = PROJECT_ROOT / "configs"
PHASE1_DIR = PROJECT_ROOT / "checkpoints" / "phase1"

# (run name, input kind, P, save reference capture for Phase 2?)
RUNS = [("text_8192", "text", 8192, False),
        ("text_2048", "text", 2048, True),
        ("random_2048", "random", 2048, False)]


def run_phase1_model(yaml_name: str, tag: str, store_device: str = "cpu") -> dict:
    final_path = PHASE1_DIR / f"report_{tag}.json"
    part_path = PHASE1_DIR / f"report_{tag}.partial.json"
    if final_path.exists() and not FORCE_RERUN:
        print(f"[1.6] {tag}: {final_path.name} exists -> skip (FORCE_RERUN=True to redo)")
        return json.loads(final_path.read_text())

    model, tok, cfg = load_falcon_h1_for_t4(CFG_DIR / yaml_name, load_tokenizer=True, device_map={"": 0})
    arch = cfg["architecture"]; L = arch["num_hidden_layers"]
    hook = FalconH1StateHook(model, store_device=store_device)
    t1_layers = sorted({0, L // 4, L // 2, (3 * L) // 4, L - 1})

    report = {"model": cfg["model_name"], "tag": tag, "L": L, "arch": arch,
              "torch": torch.__version__, "runs": {}}
    if part_path.exists() and not FORCE_RERUN:
        report["runs"] = json.loads(part_path.read_text()).get("runs", {})

    for name, kind, P, save in RUNS:
        ref_path = PHASE1_DIR / f"ref_capture_{tag}_{name}.pt"
        if name in report["runs"] and (not save or ref_path.exists()):
            print(f"[1.6] {tag}/{name}: done earlier -> skip"); continue
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(0)
        ids = make_input_ids(kind, P, model.config.vocab_size, tok, seed=1234, device="cuda:0")
        t0 = time.perf_counter()
        records, cache, logits = hook.capture(ids)
        torch.cuda.synchronize(); cap_s = time.perf_counter() - t0
        peak_gb = torch.cuda.max_memory_allocated(0) / 1e9
        rec_gb = hook.records_nbytes(records) / 1e9

        rep = verify_capture(records, hook.mixers, step_layers=None, test1_layers=t1_layers,
                             logits=logits, compute_device="cuda:0")
        s = rep["summary"]
        s.update({"P": P, "kind": kind, "capture_seconds": round(cap_s, 2),
                  "records_gb": round(rec_gb, 3), "peak_vram_gb_cuda0": round(peak_gb, 3)})
        report["runs"][name] = {"summary": s, "layers": rep["layers"]}

        if save:
            torch.save({"meta": {"model": cfg["model_name"], "tag": tag, "P": P, "kind": kind, "L": L,
                                 "layout": {"b_hat": "[B,P,d_state]", "u": "[B,P,H,d_head]", "w": "[B,P,H]",
                                            "log_alpha": "[B,P,H]", "alpha": "[B,P,H]", "S_P": "[B,H,d_state,d_head]"}},
                        "input_ids": ids.cpu(),
                        "layers": {i: {k: v.cpu() for k, v in r.items()} for i, r in records.items()}}, ref_path)
        print(f"[1.6] {tag}/{name}: P={P} pass={s['phase1_pass']} | test2 decomp max={s['test2_decomp_max']:.2e} "
              f"step max={s['test2_step_max']:.2e} | test1 {s['test1_scaled_err_max']:.2e} | finite={s['all_finite']} "
              f"| cap {cap_s:.1f}s verify {s['verify_seconds']}s | peak VRAM {peak_gb:.2f} GB | records {rec_gb:.2f} GB")
        part_path.write_text(json.dumps(report))
        del records, cache, logits, ids; gc.collect(); torch.cuda.empty_cache()

    report["phase1_pass"] = all(r["summary"]["phase1_pass"] for r in report["runs"].values())
    final_path.write_text(json.dumps(report))
    hook.detach(); del hook, model; gc.collect(); torch.cuda.empty_cache()
    return report


rep_05b = run_phase1_model("falcon_h1_0.5b.yaml", "05b")
rep_15b = run_phase1_model("falcon_h1_1.5b_deep.yaml", "15b")
print("\n[Phase 1.6] done. 0.5B pass:", rep_05b["phase1_pass"], "| 1.5B pass:", rep_15b["phase1_pass"])

Loading weights: 100%
 579/579 [00:00<00:00, 1172.79it/s]
[1.6] 05b/text_8192: P=8192 pass=True | test2 decomp max=2.62e-06 step max=5.30e-05 | test1 6.31e-07 | finite=True | cap 8.4s verify 18.37s | peak VRAM 6.52 GB | records 2.11 GB
[1.6] 05b/text_2048: P=2048 pass=True | test2 decomp max=2.80e-06 step max=2.00e-05 | test1 6.55e-07 | finite=True | cap 1.3s verify 4.18s | peak VRAM 1.50 GB | records 0.55 GB
[1.6] 05b/random_2048: P=2048 pass=True | test2 decomp max=1.77e-06 step max=1.15e-05 | test1 5.31e-07 | finite=True | cap 0.9s verify 4.14s | peak VRAM 1.50 GB | records 0.55 GB
tokenizer_config.json: 100%
 99.9k/99.9k [00:00<00:00, 9.13MB/s]
tokenizer.json: 100%
 5.06M/5.06M [00:00<00:00, 69.5MB/s]
special_tokens_map.json: 100%
 7.69k/7.69k [00:00<00:00, 771kB/s]
chat_template.jinja: 100%
 1.15k/1.15k [00:00<00:00, 132kB/s]
Loading weights: 100%
 1125/1125 [00:01<00:00, 880.77it/s]
[1.6] 15b/text_8192: P=8192 pass=True | test2 decomp max=2.58e-06 step max=9.28e-05 | test1 1.38e-

### Cell 1.7 - PHASE 1 CHECK CELL (validates outputs, writes manifest, prints what to send back) | Phase 1: State Hook & Canonicalization

In [8]:
#Cell 1.7 - PHASE 1 CHECK CELL (validates outputs, writes manifest, prints what to send back) | Phase 1: State Hook & Canonicalization
import json, torch, transformers
from pathlib import Path
PROJECT_ROOT = Path("/kaggle/working/hybrid_state_eviction_v13")
PHASE1_DIR = PROJECT_ROOT / "checkpoints" / "phase1"

problems, summary_out = [], {}
for tag, L_expected, dstate in (("05b", 36, 128), ("15b", 66, 256)):
    rp = PHASE1_DIR / f"report_{tag}.json"
    if not rp.exists():
        problems.append(f"missing {rp.name}"); continue
    rep = json.loads(rp.read_text())
    if rep["L"] != L_expected: problems.append(f"{tag}: L={rep['L']} != {L_expected}")
    if rep["arch"]["mamba_d_state"] != dstate: problems.append(f"{tag}: d_state mismatch")
    summary_out[tag] = {}
    for name, r in rep["runs"].items():
        s = r["summary"]
        summary_out[tag][name] = {k: s[k] for k in ("P", "n_layers", "test2_decomp_max", "test2_step_max",
                                  "test1_scaled_err_max", "all_finite", "ranges_ok", "phase1_pass",
                                  "capture_seconds", "records_gb", "peak_vram_gb_cuda0")}
        if s["n_layers"] != L_expected: problems.append(f"{tag}/{name}: n_layers={s['n_layers']}")
        if not s["test2_pass"]:     problems.append(f"{tag}/{name}: Test 2 FAIL (decomp {s['test2_decomp_max']:.2e}, step {s['test2_step_max']})")
        if not s["test1_pass"]:     problems.append(f"{tag}/{name}: Test 1 FAIL ({s['test1_scaled_err_max']:.2e})")
        if not s["stability_pass"]: problems.append(f"{tag}/{name}: stability FAIL (finite={s['all_finite']}, ranges={s['ranges_ok']})")
    ref = PHASE1_DIR / f"ref_capture_{tag}_text_2048.pt"
    if not ref.exists():
        problems.append(f"missing {ref.name}")
    else:
        blob = torch.load(ref, map_location="cpu")
        assert len(blob["layers"]) == L_expected, f"{tag}: ref capture has {len(blob['layers'])} layers"
        l0 = blob["layers"][0]; P = blob["meta"]["P"]
        exp = {"b_hat": (1, P, dstate), "u": (1, P, 24, 64), "w": (1, P, 24), "log_alpha": (1, P, 24),
               "alpha": (1, P, 24), "S_P": (1, 24, dstate, 64)}
        for k, shp in exp.items():
            if tuple(l0[k].shape) != shp: problems.append(f"{tag}: {k} shape {tuple(l0[k].shape)} != {shp}")
            if l0[k].dtype != torch.float32: problems.append(f"{tag}: {k} dtype {l0[k].dtype}")
        del blob

manifest = {"phase": 1, "torch": torch.__version__, "transformers": transformers.__version__,
            "handover_dir": str(PHASE1_DIR),
            "files": sorted(p.name for p in PHASE1_DIR.iterdir()),
            "hook_module": "hooks/falcon_h1_state_hook.py", "checks_module": "hooks/phase1_checks.py",
            "contract_keys": ["b_hat", "u", "w", "log_alpha", "alpha", "S_P"], "extras": ["delta", "b_norm"],
            "problems": problems}
(PHASE1_DIR / "phase1_manifest.json").write_text(json.dumps(manifest, indent=1))

print("=" * 70)
print("PHASE 1 SUMMARY (paste everything between the lines back to me)")
print("=" * 70)
print(json.dumps(summary_out, indent=1))
print("problems:", problems if problems else "NONE")
print("=" * 70)
print("PHASE 1 GATE:", "PASS" if not problems else "FAIL")

PHASE 1 SUMMARY (paste everything between the lines back to me)
{
 "05b": {
  "text_8192": {
   "P": 8192,
   "n_layers": 36,
   "test2_decomp_max": 2.6244198196263277e-06,
   "test2_step_max": 5.2973746073328696e-05,
   "test1_scaled_err_max": 6.305927847344264e-07,
   "all_finite": true,
   "ranges_ok": true,
   "phase1_pass": true,
   "capture_seconds": 8.39,
   "records_gb": 2.106,
   "peak_vram_gb_cuda0": 6.52
  },
  "text_2048": {
   "P": 2048,
   "n_layers": 36,
   "test2_decomp_max": 2.8004889443809444e-06,
   "test2_step_max": 2.0021260287095426e-05,
   "test1_scaled_err_max": 6.546528738844567e-07,
   "all_finite": true,
   "ranges_ok": true,
   "phase1_pass": true,
   "capture_seconds": 1.3,
   "records_gb": 0.548,
   "peak_vram_gb_cuda0": 1.497
  },
  "random_2048": {
   "P": 2048,
   "n_layers": 36,
   "test2_decomp_max": 1.7690095758538785e-06,
   "test2_step_max": 1.1465820812928693e-05,
   "test1_scaled_err_max": 5.31049226882604e-07,
   "all_finite": true,
   "ranges_o